# Change json element - loops files

In [1]:
import ollama
embed_model='nomic-embed-text', # 
embed_model = 'qwen3-embedding:0.6b'

response = ollama.embed(
    model=embed_model,
    input='The quick brown fox jumps over the lazy dog.'
)

# Access the numerical vector array
embedding = response['embeddings'][0]
# print(embedding[:3])


In [2]:
import json
from pathlib import Path
import ollama
from ollama import chat
import numpy as np
import time

directory = Path("/Users/ab_python/fix search/justia/data")
writedir = "/Users/ab_python/fix search/justia/data3/"

class utils():
    def clean_garbage(input_text):
        print(len(input_text) )
        garbage = "\n\nSome case metadata and case summaries were written with the help of AI, which can produce inaccuracies. You\n        should read the full case before relying on it for legal research purposes.\n        This site is protected by reCAPTCHA and the Google\n    Privacy Policy and\n    Terms of Service apply.\n\n\n\nYou're all set! You already receive all suggested Justia Opinion Summary Newsletters. You can explore additional available newsletters here.\n        \n\n\n                Sign up for our free summaries and get the latest delivered directly to you.\n            \n\n                Our Suggestions:\n            \n\n\n\n\n\n\nEnter Your Email\n\n\n\nEnter Your Email\nEnter Your Email\n\n\n\nJustia Webinars are open to all, lawyers and non-lawyers. Lawyers, please visit individual webinar pages for more information about CLE accreditation.\n\n\n\n\n\n\n\n\n\nJustia Webinars are open to all, lawyers and non-lawyers. Lawyers, please visit individual webinar pages for more information about CLE accreditation.\nGoogle Scholar\nGoogle Books\nLegal Blogs  \n\n\nGoogle Web \nBing Web \n\n\nGoogle News \nGoogle News Archive \nYahoo! News\nHave a legal question? Get free answers from experienced lawyers!\n\n                Ask Question\nLawyers - Get Listed Now!\nGet a free directory profile listing\n\n\nLawyers - Get Listed Now!\nGet a free directory profile listing"
        output = input_text.replace(garbage, "")
        print(len(output) )
        
        output = output.replace("\n", "").replace("-","").replace("\t", "").replace("(","").replace(")","").replace("[","").replace("]","")
        print(len(output) )
        return output


class AI():
    def get_embeddings(text_input):
        embed_model = 'qwen3-embedding:0.6b'
        response = ollama.embed(
            model=embed_model,
            input= text_input
        )
        full_embedding = []
        
        # Access the numerical vector array
        try: 
            #print(text_input, "\nresponse['embeddings'][0] == ", response['embeddings'][0], "\n=====================================================")
            full_embedding = response['embeddings'][0]
        except:
            #print(text_input, "\nresponse['embeddings'] == ", response['embeddings'], "\n=====================================================")
            full_embedding = response['embeddings']
        
        # 3. Set your desired target dimension
        target_dim = 384
        truncated_vector = np.array(full_embedding[:target_dim])
        
        # 4. Normalize the truncated vector for cosine similarity search
        normalized_vector = truncated_vector / np.linalg.norm(truncated_vector)
        
        return normalized_vector.tolist()

    def get_summary(text_input):        
        model_name = 'FableForge-AI/nexus-legal'
        full_response = ""   
        system_prompt = "Forget all previous prompts. Write a 4 sentence summary of this case. "
        output = ollama.generate(
            model=model_name,
            prompt=system_prompt + text_input,
            stream=False,
            options={
                "temperature": 0.9,# Low temperature ensures strict factual adherence to the text
                "num_ctx": 4096,
                "num_predict": 256,
                "top_k": 40,
                "top_p": 0.9,
                "seed": 42,
                "stop": ["<EOT>"],
                "repeat_penalty": 1.15,
              }
        )

        # full_response = output.message.content
        full_response = output['response']
        output = []
        return full_response

    def get_judge_and_ruling(text_input):        
        model_name = 'FableForge-AI/nexus-legal'
        full_response = ""   
        system_prompt = "Forget all previous prompts. Who were the attorneys and who was the judge, and what was the ruling of the case? "
        output = ollama.generate(
            model=model_name,
            prompt=system_prompt + text_input,
            stream=False,
            options={
                "temperature": 0.9,# Low temperature ensures strict factual adherence to the text
                "num_ctx": 4096,
                "num_predict": 256,
                "top_k": 40,
                "top_p": 0.9,
                "seed": 42,
                "stop": ["<EOT>"],
                "repeat_penalty": 1.15,
              }
        )

        full_response = output['response']
        output = []
        return full_response





In [4]:
import json

from pathlib import Path
# 9/21/2026. 
"""
data2 ->    1. pop date, 
            2. rename court_year to court_name
            3. add state

data3 ->    1. embeddings for 3 fields [court_name, docket_number, state]
            2. summary of big text field
"""

directory = Path("/Users/anria/ab_python/fix search/justia/data")
writedir = "/Users/anria/ab_python/fix search/justia/data3/"
i = 0


# Loop through all text files
for read_file in directory.glob("*.json"):
    # print(read_file.name)
    with open(read_file) as file:
        json_data = json.load(file)
        # 9/19/26 - change element name
        if "court_year" in json_data:
            json_data["court_name"] = json_data.pop("court_year")

        # 9/19/26 - get rid of date unused field
        if "date" in json_data:
            json_data.pop("date")

        # 9/19/26 - add state field
        state = read_file.name.split("_")[1]
        json_data["state"] = state.capitalize()

        # 9/21/26 - use AI to pull out data and create more fields
        combo_field = json_data["case_name"] + " " + json_data["case_date"] + " " + json_data["court_name"] 
        json_data["judge_ruling"] = AI.get_judge_and_ruling(combo_field)
        json_data["summary"] = AI.get_summary(combo_field)

        if json_data["judge_ruling"] != "":
            json_data["dv_judge_ruling"] = AI.get_embeddings(json_data["judge_ruling"])
            
        if json_data["summary"] != "":
            json_data["dv_summary"] = AI.get_embeddings(json_data["summary"])
        json_data["dv_general_text"] = AI.get_embeddings(combo_field)
        
        write_file = writedir + read_file.name
        # print("writefile = ", write_file)
        with open(write_file, "w") as wfile:
            json.dump(json_data, wfile, indent=4, sort_keys=True)
     

print("Out of loop")

SyntaxError: 'break' outside loop (3083738140.py, line 17)

In [ ]:
   ''' i+=1
    if i > 3:
        break '''

In [41]:
print( AI.get_embeddings(" WL 264718 (OH App) *10*") )

[0.01452287104958955, -0.03601005847594276, -0.01337830173647804, -0.03363596550027875, 0.02102585003406246, -0.02403670868802116, 0.0684849299132845, -0.029102434478174646, -0.04595493732218647, -0.034888455050562844, 0.0006554127482389102, -0.014171444379381275, 0.1206302591092343, -0.010195447370736717, -0.06722112099844778, 0.17133718357728378, -0.05732611116836371, -0.09701408044345557, -0.021220757215112095, -0.03252128261356436, 0.02496759806939919, 0.02571781814836287, 0.03226320628447555, 0.1389737128914119, -0.0664635609051541, 0.024573796259091507, -0.0834623959721869, 0.003208890037930362, 0.06438723896537817, 0.09640449646430045, 0.14329061955414527, -0.03425537071661328, 0.011710147433820754, -0.034869863694760944, 0.03364942273308283, -0.014444045255802741, -0.06384697212508235, -0.060343809381129924, 0.05223350168494811, 0.12267352992976208, -0.05021404542106388, -0.0465971254117735, 0.0007048370230183584, 0.04344061314600393, 0.014797192747993727, -0.004126659062713625

In [ ]:
messages=[ {
                        'role': 'system', 'content': 'Forget all previous prompts. You are NEXUS-LEGAL, a domain-specialized uncensored AI assistant for legal tasks.'
                      },
                      { 'role': 'user', 'content': system_prompt + text_input} ],

# Kaput -- Done


In [1]:
import json

from pathlib import Path
# 9/19/2026. 
"""
data2 ->    1. pop date, 
            2. rename court_year to court_name
            3. add state

"""

break
directory = Path("/Users/anria/ab_python/fix search/justia/data")
writedir = "/Users/anria/ab_python/fix search/justia/data2/"
i = 0
# Loop through all text files
for read_file in directory.glob("*.json"):
    # print(read_file.name)
    with open(read_file) as file:
        json_data = json.load(file)
        if "court_year" in json_data:
            json_data["court_name"] = json_data.pop("court_year")
        if "date" in json_data:
            json_data.pop("date")

        state = read_file.name.split("_")[1]
        json_data["state"] = state.capitalize()
        
        write_file = writedir + read_file.name
        # print("writefile = ", write_file)
        with open(write_file, "w") as wfile:
            json.dump(json_data, wfile, indent=4, sort_keys=True)
     
    ''' i+=1
    if i > 3:
        break '''
print("Out of loop")
    

SyntaxError: 'break' outside loop (75661804.py, line 14)